[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ricalanis/openenv-hackaton/blob/main/training/train_enrichment.ipynb)

# DataSage Stage 2: Enrichment GRPO Training

Pre-builds dataset with real environment observations (no `rollout_func`).
Reward functions replay the env with stored seeds for context-matched evaluation.

**Fully self-contained** — runs in Colab with no repo clone or project imports.

**Requirements:** GPU (A100/H100), `HF_TOKEN`, `WANDB_API_KEY`.

In [ ]:
# Setup: install dependencies
!pip install -q unsloth trl datasets wandb requests

In [ ]:
# Load API keys from Colab Secrets or set manually
import os

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    print("Loaded keys from Colab Secrets")
except Exception:
    pass  # Fill manually below if not using Colab Secrets

# Uncomment and fill if not using Colab Secrets:
# os.environ["HF_TOKEN"] = "hf_..."
# os.environ["WANDB_API_KEY"] = "wandb_..."

assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in Colab Secrets or above"
print("Keys loaded.")

In [ ]:
# Config + parser + reward helpers (all inlined, no project imports)
import json
import re
import requests

# ---- Config constants ----
WANDB_PROJECT = "datasage"
BASE_MODEL = "unsloth/Qwen2.5-3B-Instruct"
ENV_URL = "https://ricalanis-datasage-enrichment.hf.space"
HF_REPO = "ricalanis/datasage-qwen-enrichment"

LEARNING_RATE = 5e-6
NUM_TRAIN_EPOCHS = 3
PER_DEVICE_TRAIN_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
NUM_GENERATIONS = 4
MAX_COMPLETION_LENGTH = 256
MAX_PROMPT_LENGTH = 512


# ---- Completion text extraction ----
def _get_text(completion) -> str:
    """Extract text from a completion (str or chat message list)."""
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list):
        # Chat format: [{"role": "assistant", "content": "..."}]
        return completion[-1]["content"] if completion else ""
    return str(completion)


# ---- Parser ----
def parse_enrichment_action(text: str) -> dict:
    """Parse model output into an enrichment action dict."""
    json_match = re.search(r'\{[^{}]*"operation"[^{}]*\}', text, re.DOTALL)
    if json_match:
        try:
            data = json.loads(json_match.group())
            if "field_name" in data:
                return data
        except json.JSONDecodeError:
            pass

    text_lower = text.lower()
    sources = [
        "salary_band", "tenure_risk", "satisfaction_index", "industry_benchmark",
        "flight_risk_score", "deal_size_category", "velocity_score",
        "win_probability_model", "industry_code", "competitive_risk",
        "schedule_risk_score", "resource_utilization", "dependency_chain_depth",
        "burndown_rate", "delay_probability", "sla_compliance_flag", "mttr_band",
        "escalation_path", "incident_severity_score", "recurring_pattern_flag",
    ]
    for source in sources:
        if source.replace("_", " ") in text_lower or source in text_lower:
            return {"operation": "add_field", "field_name": source, "source": source,
                    "logic": "", "params": {}}

    return {"operation": "add_field", "field_name": "unknown", "source": "", "logic": "", "params": {}}


# ---- Reward helpers ----
_ACTION_JSON_RE = re.compile(r'\{[^{}]*"operation"[^{}]*\}')
ENRICHMENT_VALID_OPS = {"add_field", "lookup", "compute_derived", "add_category"}


def source_relevance_reward(completions, **kwargs) -> list[float]:
    """Reward picking a valid enrichment source for the domain."""
    sources_list = kwargs.get("available_sources", [])
    if not sources_list:
        return [0.0] * len(completions)
    rewards = []
    for completion, available in zip(completions, sources_list):
        text = _get_text(completion)
        action_dict = parse_enrichment_action(text)
        source = action_dict.get("source", "")
        if source in available:
            rewards.append(1.0)
        elif action_dict.get("field_name", "unknown") != "unknown":
            rewards.append(0.3)
        else:
            rewards.append(0.0)
    return rewards


def enrichment_json_format_reward(completions, **kwargs) -> list[float]:
    """Reward well-formed JSON enrichment actions."""
    rewards = []
    for completion in completions:
        text = _get_text(completion)
        match = _ACTION_JSON_RE.search(text)
        if match:
            try:
                data = json.loads(match.group())
                op_ok = data.get("operation") in ENRICHMENT_VALID_OPS
                field_ok = "field_name" in data and data["field_name"] != "unknown"
                if op_ok and field_ok:
                    rewards.append(1.0)
                elif op_ok:
                    rewards.append(0.5)
                else:
                    rewards.append(0.2)
            except (json.JSONDecodeError, AttributeError):
                rewards.append(0.2)
        else:
            rewards.append(0.0)
    return rewards


print(f"Environment URL: {ENV_URL}")
print(f"Base model: {BASE_MODEL}")

In [ ]:
# Model loading via Unsloth
import torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=2048, load_in_4bit=True,
    dtype=torch.float16,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Model loaded: {BASE_MODEL}")

In [ ]:
# System prompt and task descriptions
SYSTEM_PROMPT = """\
You are a data enrichment agent. You enrich enterprise datasets by adding \
derived fields, lookups, and computed columns across multiple domains \
(HR, Sales, Project Management, IT Operations).

Analyze the schema and available sources below, then respond with a JSON \
enrichment action:
{"operation": "<op>", "field_name": "<name>", "source": "<source>", "logic": "<logic>", "params": {}}

Available operations:
- add_field: Add a new enrichment field from a known source
- lookup: Look up external reference data
- compute_derived: Compute a derived metric from existing columns
- add_category: Add a categorical classification

Identify the most valuable enrichment to add and act."""

TASK_DESCRIPTIONS = [
    "Enrich this dataset by adding the most valuable derived field.",
    "Add an enrichment that increases analytical coverage the most.",
    "Look at the available sources and add the most impactful one.",
    "This dataset needs enrichment. Choose the best source to add.",
    "Maximize enrichment coverage by adding the most useful field.",
    "Analyze the schema and pick the enrichment with highest value.",
    "Add a derived field that enables the most downstream analysis.",
    "Choose an enrichment source that fills the biggest analytics gap.",
]

In [ ]:
# Dataset: pre-built with real environment observations
import random
from datasets import Dataset

random.seed(42)
SEEDS = [42 + i for i in range(64)]
prompts = []
available_sources_list = []

print("Building dataset with real environment observations...")
for i, seed in enumerate(SEEDS):
    task_desc = random.choice(TASK_DESCRIPTIONS)
    resp = requests.post(f"{ENV_URL}/web/reset", json={"seed": seed})
    resp.raise_for_status()
    obs = resp.json()["observation"]

    prompts.append([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": (
            f"Domain: {obs['domain']}\n\nSchema:\n{obs['schema_info']}\n\n"
            f"Available Enrichment Sources: {', '.join(obs['available_sources'])}\n\n"
            f"Possible Enrichments: {', '.join(obs['possible_enrichments'])}\n\n"
            f"Data Preview:\n{obs['data_preview']}\n\nTask: {task_desc}"
        )},
    ])
    available_sources_list.append(obs["available_sources"])
    if (i + 1) % 16 == 0:
        print(f"  Built {i + 1}/64 examples")

dataset = Dataset.from_dict({
    "prompt": prompts,
    "seed": SEEDS,
    "available_sources": available_sources_list,
})
print(f"Dataset size: {len(dataset)}")

In [ ]:
# Environment reward function (calls env directly with stored seeds)
def enrichment_env_reward(completions, **kwargs) -> list[float]:
    """Primary reward: replay env with stored seed, step with parsed action."""
    seeds = kwargs.get("seed", [0] * len(completions))
    rewards = []
    for completion, seed in zip(completions, seeds):
        try:
            text = _get_text(completion)
            action_dict = parse_enrichment_action(text)
            requests.post(f"{ENV_URL}/web/reset", json={"seed": int(seed)})
            resp = requests.post(f"{ENV_URL}/web/step", json={"action": action_dict})
            resp.raise_for_status()
            rewards.append(float(resp.json().get("reward", 0.0)))
        except Exception as e:
            print(f"Env error: {e}")
            rewards.append(0.0)
    return rewards

In [ ]:
# GRPO training config
from trl import GRPOConfig

training_args = GRPOConfig(
    output_dir="./outputs/enrichment-grpo",
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_COMPLETION_LENGTH,
    max_prompt_length=MAX_PROMPT_LENGTH,
    logging_steps=1, save_steps=50, fp16=True, bf16=False,
    report_to="wandb", run_name="datasage-enrichment-grpo-v2",
)
os.environ.setdefault("WANDB_PROJECT", WANDB_PROJECT)

In [ ]:
# Create trainer and train
from trl import GRPOTrainer

trainer = GRPOTrainer(
    model=model, processing_class=tokenizer, args=training_args,
    train_dataset=dataset,
    reward_funcs=[enrichment_env_reward, source_relevance_reward, enrichment_json_format_reward],
)
print("Starting Stage 2 (Enrichment) GRPO training...")
trainer.train()

In [ ]:
# Save and push to Hub
output_dir = "./outputs/enrichment-grpo-final"
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Model saved to {output_dir}")

print(f"Pushing to Hub: {HF_REPO}")
trainer.push_to_hub(HF_REPO)
print(f"Model pushed to https://huggingface.co/{HF_REPO}")